## 爬蟲：捕捉資料
新聞標題的 class (分類) 是根據網頁的原始碼所給定的。

In [ ]:
# 自由時報資料抓取 (一年)
import requests, time, os, random
import pandas as pd
from bs4 import BeautifulSoup

file_path = "run_results/ltn1-1.csv"

# -----------------------------
# 設定
# -----------------------------
keywords = ["生活", "財經", "社會", "政治", "國際"]
allowed_classes = keywords  # 僅允許的 class
start_date = "20250101"
end_date = "20250228"
clear_file = True  # True 表示清空 CSV 後抓取，False 表示累計舊資料

# -----------------------------
# 初始化 CSV 與 existing_titles
# -----------------------------
existing_titles = set()
if os.path.exists(file_path) and not clear_file:
    df_existing = pd.read_csv(file_path, encoding='utf-8-sig')
    existing_titles.update(df_existing['title'].tolist())
else:
    # 清空檔案並寫入欄位
    df_empty = pd.DataFrame(columns=['title','class','time','link'])
    df_empty.to_csv(file_path, index=False, header=True, encoding='utf-8-sig')

total_new_count = 0

# -----------------------------
# 開始抓取
# -----------------------------
for keyword in keywords:
    print(f"\n開始抓取關鍵字：{keyword}")
    page = 1
    while True:
        # 隨機 sleep 1~3 秒
        time.sleep(random.uniform(1,3))
        url = f"https://search.ltn.com.tw/list?keyword={keyword}&start_time={start_date}&end_time={end_date}&page={page}"
        print(f"抓取第 {page} 頁：{url}")

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
        }
        response = requests.get(url, headers=headers)
        response.encoding = 'utf-8'

        if response.status_code != 200:
            print("網頁抓取失敗，跳出關鍵字抓取。")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        objTag = soup.find_all('div', class_='cont')

        if not objTag:
            print("本頁沒有新聞，結束本關鍵字抓取。")
            break

        rows_to_add = []
        page_new_count = 0

        for data in objTag:
            try:
                title = data.find(class_="tit").get_text().strip()
                link = data.find('a', class_="tit").get('href')
            except AttributeError:
                continue  # 若標題或連結抓不到就跳過

            # 已存在就跳過
            if title in existing_titles:
                continue

            # 取 class
            class1 = ''
            try:
                # 還有 immtag chan2、immtag chan7、immtag chan11 ... 等
                class1 = data.find(class_="immtag chan").get_text().strip()
            except AttributeError:
                continue  # 無 class 跳過

            # class 必須在允許列表
            if class1 not in allowed_classes:
                continue

            # 取時間
            try:
                time1 = data.find('span', class_="time").get_text().strip()
            except AttributeError:
                time1 = ""

            rows_to_add.append([title, class1, time1, link])
            existing_titles.add(title)
            page_new_count += 1
            total_new_count += 1

        # 批次寫入 CSV
        if rows_to_add:
            df_page = pd.DataFrame(rows_to_add, columns=['title','class','time','link'])
            df_page.to_csv(file_path, mode='a', index=False, header=False, encoding='utf-8-sig')

        print(f"本頁新增 {page_new_count} 篇新聞。")

        # 如果本頁沒有新增新聞，表示已經抓完
        if page_new_count == 0:
            print("本頁全部新聞已存在或不符合條件，結束本關鍵字抓取。")
            break

        page += 1


print(f"\n共新增 {total_new_count} 筆新聞到 {file_path}")


開始抓取關鍵字：生活
抓取第 1 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=1
本頁新增 11 篇新聞。
抓取第 2 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=2
本頁新增 10 篇新聞。
抓取第 3 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=3
本頁新增 9 篇新聞。
抓取第 4 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=4
本頁新增 7 篇新聞。
抓取第 5 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=5
本頁新增 10 篇新聞。
抓取第 6 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=6
本頁新增 8 篇新聞。
抓取第 7 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=7
本頁新增 9 篇新聞。
抓取第 8 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=8
本頁新增 8 篇新聞。
抓取第 9 頁：https://search.ltn.com.tw/list?keyword=生活&start_time=20250301&end_time=20250430&page=9
本頁新增 11 篇新聞。
抓取第 10 頁：https://sear

In [7]:
# 查看共取得多少筆資料
df_all = pd.read_csv("run_results/ltn1.csv")
print("Data in CSV:", len(df_all))

Data in CSV: 17032


## 2. 情感分析 - 方法
### (1) 自定義情緒詞典規則法
加入自定義的情緒標籤：ltn1_sentiment1.csv

In [1]:
import pandas as pd
import jieba
df = pd.read_csv("run_results/ltn1a.csv")

positive_words = {
    "好", "提升", "成長", "強", "讚", "創新", "贏", "熱烈", "突破", "進補",
    "進步", "進展", "進化", "進取", "進攻", "優化", "優良", "優秀", "優勢", "優越", "優異",
    "優良", "優質", "優渥", "正向", "積極", "攻頂", "觀光", "遊憩", "合作", "智能客服", "無人機", "揭露",
    "喜悅", "支持", "振興", "利多", "增長", "發展", "成功", "榮耀", "順利", "改善",
    "勝利", "穩定", "光明", "鼓舞", "幸福", "成就", "突破性", "引領", "佳績", "樂觀", "好轉"
}
negative_words = {
    "壞", "下滑", "崩盤", "惡化", "危機", "損失", "爭議", "跌", "惱人", "過時", "衰退",
    "失靈", "失準", "失誤", "失落", "失望", "失敗", "失控", "失業", "失信", "失去", "危險", "危害", "危險性", 
    "風險", "風暴", "風波", "風險性", "恐怖", "恐慌", "恐懼", "恐嚇", "恐怖主義", "恐怖分子", "恐怖攻擊", "恐怖活動",
    "恐怖份子", "恐怖襲擊", "恐怖事件", "恐怖威脅", "恐怖分子活動", "恐怖主義者", "恐怖分子襲擊", "恐怖分子威脅", "恐怖分子攻擊",
    "恐怖分子事件", "震盪", "滅絕", "暴政", "收黑", "拖累", "困難", "失敗", "警告", "危險", "疲弱",
    "衰退", "減少", "受挫", "困境", "混亂", "損害", "損失慘重", "失控", "恐慌", "糟糕",
    "不穩", "破產", "倒閉", "惡劣", "隱憂", "低迷", "危殆", "災難", "退步", "損傷", "不好"
}


def simple_sentiment(text):
    tokens = jieba.lcut(str(text))
    pos = sum(1 for w in tokens if w in positive_words)
    neg = sum(1 for w in tokens if w in negative_words)
    if pos > neg:
        return "正面"
    elif neg > pos:
        return "負面"
    else:
        return "中立"
df["sentiment"] = df["title"].apply(simple_sentiment)
df.to_csv("run_results/ltn1_sentiment1.csv", index=False, encoding="utf-8-sig")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\yjche\AppData\Local\Temp\jieba.cache
Loading model cost 1.114 seconds.
Prefix dict has been built successfully.


### (2) 套用預訓練的深度學習模型
使用預訓練的中文情感分析模型，對 CSV 檔案中的新聞標題進行情緒分類，並將結果存回 CSV。

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
df = pd.read_csv("ltn1a.csv")
if "title" not in df.columns:
    raise ValueError("CSV 檔案中缺少 title 欄位")
# 使用預訓練模型進行情感分析：模型為 IDEA-CCNL/Erlangshen-RoBERTa-110M-Sentiment，是 中文情感分析的 RoBERTa 模型。
# 模型輸出標籤通常為 'pos'（正向）、'neg'（負向）、'neu'（中立）。
classifier = pipeline("text-classification", model="IDEA-CCNL/Erlangshen-RoBERTa-110M-Sentiment")
def analyze_sentiment(text):
    try:
        result = classifier(str(text)[:512])[0] 
        return result["label"]  # 'pos', 'neg', 'neu'
    except:
        return "error"
tqdm.pandas(desc="分析情緒中")
df["sentiment"] = df["title"].progress_apply(analyze_sentiment)
df.to_csv("runs_results/ltn1_sentiment2.csv", index=False, encoding="utf-8-sig")
print("分析完成，已儲存為 ltn1_sentiment2.csv")


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

c:\Users\yjche\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yjche\.cache\huggingface\hub\models--IDEA-CCNL--Erlangshen-RoBERTa-110M-Sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/409M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu
分析情緒中: 100%|██████████| 29682/29682 [37:06<00:00, 13.33it/s] 


分析完成，已儲存為 ltn1_sentiment2.csv
